In [ ]:
# llm-based parallel workflow
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from pydantic import BaseModel, Field
import operator

In [ ]:
load_dotenv()

In [ ]:
model = ChatOpenAI(model='gpt-4o-mini')

In [ ]:
class EvaluationSchema(BaseModel):
    feedback:str = Field(description='Detailed feedbackfor the easy')
    score: int = Field(description='Score out of 10', ge=0, le=10)

In [ ]:
structured_model = model.with_structured_output(EvaluationSchema)

In [ ]:
essay = """
    Human life is full of decisions. From a student choosing a career to a government deciding a policy, every decision involves some amount of uncertainty. Sometimes we fear making the wrong decision so much that we choose not to act at all. However, in many situations, doing nothing can be more harmful than taking a decision and making a mistake.

Being wrong is a natural part of learning. A person who tries something new may fail, but that failure provides experience. For example, a student preparing for a competitive examination may choose an ineffective study strategy. If the student realizes the mistake and changes the strategy, the failure becomes a learning experience. But if the student keeps postponing preparation because of the fear of failure, valuable time is permanently lost.

The same principle applies to entrepreneurship and innovation. Many successful businesses and inventions emerged after several failures. Thomas Edison, for example, conducted numerous experiments before developing a successful electric light bulb. His failures were not completely useless because they helped him understand what did not work. Had he decided not to experiment because he feared failure, innovation would have been delayed.

At the level of society, the consequences of inaction can be even greater. Consider environmental problems. If governments wait for perfect scientific certainty before taking action against climate change, the damage to ecosystems may become irreversible. Similarly, during a public-health crisis, delaying necessary action can allow a problem to grow rapidly. Therefore, responsible decision-making sometimes requires acting despite incomplete information.

However, the statement does not mean that every decision should be taken without thinking. There is a difference between courageous action and reckless action. Decisions should be based on available evidence, consultation, risk assessment and ethical considerations. The goal should not be to eliminate mistakes completely, because that is impossible, but to make mistakes that are manageable and reversible whenever possible.

Doing nothing also has a hidden cost. When a person avoids making a decision, circumstances continue to change. Opportunities may disappear, problems may become larger, and the eventual cost of action may become much higher. For example, a government that delays reforms because they may face criticism today might eventually have to implement much more difficult reforms tomorrow.

In personal life too, hesitation can prevent growth. A person may avoid learning a new skill because they are afraid of failing. They may avoid starting a business because they fear financial loss. They may avoid expressing an important idea because they fear criticism. In all these situations, failure is possible, but inaction guarantees that the desired outcome will not even be attempted.

At the same time, society must create an environment where honest mistakes are treated as opportunities for learning rather than reasons for punishment. Scientists, entrepreneurs, administrators and students should be encouraged to experiment responsibly. A culture that punishes every failure creates fear, while a culture that learns from failure creates innovation.

Thus, the real challenge is not to choose between action and mistakes blindly. It is to make informed decisions, accept reasonable risks, learn from mistakes and continuously improve. In many situations, a wrong decision can be corrected, but the opportunity lost through prolonged inaction may never return.

Therefore, the cost of being wrong is often temporary, while the cost of doing nothing can sometimes be permanent. Progress—whether personal, social or national—requires the courage to act, the wisdom to evaluate our mistakes and the humility to learn from them.
"""

prompt = f"Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10 \n {essay}"
structured_model.invoke(prompt)

In [ ]:
class UPSEState(TypedDict):
    essay: str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str
    # operator.add add the output of each node to the list
    # [op1], [op2], [op2] -> [op1, op2, op3]
    individual_scores: Annotated[list[int], operator.add]
    avg_score: float

In [ ]:
def evaluate_language(state: UPSEState):
    prompt = f"Evaluate the depth of analysis of the following essay and provide a feedback and provide a feedback and   of 10 \n {essay}"
    output = structured_model.invoke(prompt)

    return {'analysis_feedback': output.feedback, 'Individual_score': [output.score]}

def evaluate_analysis(state: UPSEState):
    prompt = f"Evaluate the clarity of thought of the following essay and provide a feedback and provide a feedback and   of 10 \n {essay}"
    output = structured_model.invoke(prompt)

    return {'analysis_feedback': output.feedback, 'Individual_score': [output.score]}

In [ ]:
graph = StateGraph(UPSEState)

graph.add_node('evaluate_language', evaluate_language)
graph.add_node('evaluate_analysis', evaluate_analysis)
graph.add_node('evaluate_thought', evaluate_thought)
graph.add_node('final_evaluation', final_evaluation)